# AutoEncoder

## Dataset Creation

In [2]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np

# Root of this repo (assumes this notebook lives at repo root)
REPO_ROOT = Path.cwd()
PROCESSED_DIR = REPO_ROOT / "processed_data"

# Subject IDs present in processed_data/
SUBJECTS = ["CSI1", "CSI2", "CSI3", "CSI4"]

# Common Schaefer parcellations in this repo
SCHAEFER_VARIANTS = ["schaefer400", "schaefer414", "schaefer1014"]

# Safer default: prefer NOT unpickling. If an npz contains object arrays,
# we'll either (a) skip those keys or (b) enable pickle if requested.
ALLOW_PICKLE = False
SKIP_OBJECT_ARRAYS = True

print("REPO_ROOT:", REPO_ROOT)
print("PROCESSED_DIR exists:", PROCESSED_DIR.exists())
print("Subjects:", SUBJECTS)
print("Schaefer variants:", SCHAEFER_VARIANTS)

def _npz_path(subject: str, variant: str) -> Path:
    """Return path to a per-subject Schaefer .npz file."""
    return PROCESSED_DIR / subject / f"{subject}_{variant}.npz"


def _visual_rois_path(subject: str) -> Path:
    """Return path to a per-subject visual ROIs .npz file."""
    return PROCESSED_DIR / subject / f"{subject}_visual_rois.npz"


def _require_exists(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected file: {path}")

def list_available_npzs(subject: str) -> dict[str, Path]:
    """List available npz files for a given subject."""
    subj_dir = PROCESSED_DIR / subject
    if not subj_dir.exists():
        return {}
    out: dict[str, Path] = {}
    for p in sorted(subj_dir.glob("*.npz")):
        out[p.name] = p
    return out


for s in SUBJECTS:
    files = list_available_npzs(s)
    print(f"\n{s}: {len(files)} npz files")
    for name in list(files)[:10]:
        print(" -", name)
    if len(files) > 10:
        print("   ...")

def load_npz(
    path: Path,
    *,
    allow_pickle: bool | None = None,
    skip_object_arrays: bool | None = None,
) -> dict[str, np.ndarray]:
    """Load npz and return a {key: array} dict (eagerly loaded).

    Some of the repo's .npz files may contain object arrays (e.g., strings).
    By default we avoid pickle; if object arrays are present we skip them.

    Set allow_pickle=True to load everything (less safe, but sometimes needed).
    """
    _require_exists(path)
    if allow_pickle is None:
        allow_pickle = ALLOW_PICKLE
    if skip_object_arrays is None:
        skip_object_arrays = SKIP_OBJECT_ARRAYS

    out: dict[str, np.ndarray] = {}
    with np.load(path, allow_pickle=allow_pickle) as f:
        for k in f.files:
            try:
                arr = f[k]
            except ValueError as e:
                # Typical: "Object arrays cannot be loaded when allow_pickle=False"
                if skip_object_arrays:
                    print(f"[load_npz] Skipping key '{k}' in {path.name}: {e}")
                    continue
                raise

            if skip_object_arrays and isinstance(arr, np.ndarray) and arr.dtype == object:
                print(f"[load_npz] Skipping object array key '{k}' in {path.name}")
                continue
            out[k] = arr
    return out


def describe_npz(npz: dict[str, np.ndarray], max_keys: int = 20) -> None:
    keys = list(npz.keys())
    print(f"Keys ({len(keys)}):", keys[:max_keys], "..." if len(keys) > max_keys else "")
    for k in keys[:max_keys]:
        a = npz[k]
        print(f"  {k:>24}  shape={a.shape}  dtype={a.dtype}")


# Pick one subject+variant to inspect first
SUBJECT = SUBJECTS[0]
VARIANT = SCHAEFER_VARIANTS[0]
PATH = _npz_path(SUBJECT, VARIANT)
print("Inspecting:", PATH)
npz0 = load_npz(PATH)
describe_npz(npz0)

def find_data_array(npz: dict[str, np.ndarray]) -> tuple[str, np.ndarray]:
    """Heuristic: find the 'main' 2D array likely holding betas/voxel/ROI features."""
    # Common candidates
    preferred = [
        "X",
        "data",
        "betas",
        "features",
        "fmri",
        "roi_data",
        "roi_betas",
    ]
    for k in preferred:
        if k in npz and npz[k].ndim >= 2:
            return k, npz[k]

    # Fallback: choose the largest numeric array
    best_k = None
    best = None
    best_size = -1
    for k, a in npz.items():
        if not isinstance(a, np.ndarray):
            continue
        if a.dtype.kind not in "iufb":
            continue
        if a.size > best_size and a.ndim >= 2:
            best_k, best, best_size = k, a, a.size
    if best_k is None:
        raise ValueError("Couldn't find a plausible numeric 2D+ data array in npz")
    return best_k, best


data_key, X = find_data_array(npz0)
print("Main data key:", data_key)
print("X:", X.shape, X.dtype)

# Basic shape contract we want to validate:
# - samples dimension == number of images/trials
# - feature dimension == parcels/ROIs/voxels
print("n_samples:", X.shape[0])
print("n_features:", X.shape[1] if X.ndim >= 2 else None)

def summarize_numeric(a: np.ndarray, name: str) -> None:
    a = np.asarray(a)
    if a.dtype.kind not in "iufb":
        print(f"{name}: non-numeric dtype={a.dtype}, skipping")
        return

    finite = np.isfinite(a)
    n_total = a.size
    n_finite = int(finite.sum())
    n_nan = int(np.isnan(a).sum())
    n_inf = int(np.isinf(a).sum())
    print(f"{name}: shape={a.shape} dtype={a.dtype}")
    print(f"  finite: {n_finite}/{n_total}  nan: {n_nan}  inf: {n_inf}")

    # Avoid huge memory for global stats on giant arrays; sample if very large
    if n_total > 5_000_000:
        flat = a.ravel()
        idx = np.random.default_rng(0).choice(flat.size, size=1_000_000, replace=False)
        s = flat[idx]
        s = s[np.isfinite(s)]
        print("  stats (sampled): min/mean/std/max =",
              float(np.min(s)), float(np.mean(s)), float(np.std(s)), float(np.max(s)))
    else:
        b = a[np.isfinite(a)]
        if b.size:
            print("  stats: min/mean/std/max =",
                  float(np.min(b)), float(np.mean(b)), float(np.std(b)), float(np.max(b)))


summarize_numeric(X, f"{SUBJECT}:{VARIANT}:{data_key}")

def validate_rowwise(a: np.ndarray, name: str, max_report: int = 10) -> None:
    """Row-wise checks: all-finite, all-zero rows, extreme norms."""
    a = np.asarray(a)
    if a.ndim != 2 or a.dtype.kind not in "iufb":
        print(f"{name}: expected numeric 2D array, got shape={a.shape}, dtype={a.dtype}")
        return

    finite_rows = np.isfinite(a).all(axis=1)
    bad_rows = np.where(~finite_rows)[0]

    row_norm = np.linalg.norm(np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0), axis=1)
    zero_rows = np.where(row_norm == 0)[0]

    print(f"{name} row-wise:")
    print(f"  rows: {a.shape[0]}  cols: {a.shape[1]}")
    print(f"  bad (non-finite) rows: {bad_rows.size}")
    print(f"  all-zero rows: {zero_rows.size}")
    if bad_rows.size:
        print("   examples:", bad_rows[:max_report])
    if zero_rows.size:
        print("   examples:", zero_rows[:max_report])


validate_rowwise(X, f"{SUBJECT}:{VARIANT}:{data_key}")

def validate_consistent_shapes(subjects: list[str], variants: list[str]) -> None:
    """Check that all subjects+variants load and have consistent feature counts."""
    results: dict[tuple[str, str], tuple[int, int]] = {}
    for s in subjects:
        for v in variants:
            p = _npz_path(s, v)
            _require_exists(p)
            npz = load_npz(p)
            k, arr = find_data_array(npz)
            if arr.ndim != 2:
                raise ValueError(f"Expected 2D for {p} key={k}, got shape={arr.shape}")
            results[(s, v)] = (arr.shape[0], arr.shape[1])

    # Report per variant across subjects
    for v in variants:
        shapes = [results[(s, v)] for s in subjects]
        n_samples_set = sorted(set(n for n, _ in shapes))
        n_feat_set = sorted(set(m for _, m in shapes))
        print(f"\nVariant {v}:")
        print("  n_samples:", n_samples_set)
        print("  n_features:", n_feat_set)
        for s in subjects:
            print(f"   {s}: {results[(s, v)]}")


validate_consistent_shapes(SUBJECTS, SCHAEFER_VARIANTS)

def load_visual_rois(subject: str) -> dict[str, np.ndarray]:
    p = _visual_rois_path(subject)
    _require_exists(p)
    return load_npz(p)


def describe_visual_rois(subject: str) -> None:
    npz = load_visual_rois(subject)
    print(f"\n{subject} visual ROIs:")
    describe_npz(npz)


for s in SUBJECTS:
    describe_visual_rois(s)

def validate_visual_rois_against_schaefer(subject: str, schaefer_variant: str) -> None:
    sch = load_npz(_npz_path(subject, schaefer_variant))
    vk, V = find_data_array(load_visual_rois(subject))
    sk, S = find_data_array(sch)

    if V.ndim != 2 or S.ndim != 2:
        print(f"{subject}: expected 2D arrays. visual={V.shape} schaefer={S.shape}")
        return

    print(f"\n{subject}: comparing sample counts")
    print("  visual:", V.shape, "key=", vk)
    print("  schaefer:", S.shape, "key=", sk)
    if V.shape[0] != S.shape[0]:
        print("  WARNING: sample dimension differs!")


for s in SUBJECTS:
    validate_visual_rois_against_schaefer(s, "schaefer400")

def quick_duplicate_check(a: np.ndarray, name: str) -> None:
    """Detect duplicated rows (exact match) on a random subset."""
    if a.ndim != 2 or a.dtype.kind not in "iufb":
        print(f"{name}: expected numeric 2D")
        return

    n = a.shape[0]
    rng = np.random.default_rng(0)
    take = min(n, 5000)
    idx = rng.choice(n, size=take, replace=False)
    sub = a[idx]

    # Hash rows by bytes (works for exact duplicates)
    row_view = np.ascontiguousarray(sub).view(np.dtype((np.void, sub.dtype.itemsize * sub.shape[1])))
    _, counts = np.unique(row_view, return_counts=True)
    dup_rows = int(counts[counts > 1].sum())
    print(f"{name}: checked {take} rows, rows in duplicate groups={dup_rows}")


quick_duplicate_check(X, f"{SUBJECT}:{VARIANT}:{data_key}")

print("Done: core data validation cells loaded.")
print("Next: if you tell me which label keys you expect, we can validate labels ↔ samples alignment.")


REPO_ROOT: /home/mariop/Documents/Programming/bold5000-gan
PROCESSED_DIR exists: True
Subjects: ['CSI1', 'CSI2', 'CSI3', 'CSI4']
Schaefer variants: ['schaefer400', 'schaefer414', 'schaefer1014']

CSI1: 4 npz files
 - CSI1_schaefer1014.npz
 - CSI1_schaefer400.npz
 - CSI1_schaefer414.npz
 - CSI1_visual_rois.npz

CSI2: 4 npz files
 - CSI2_schaefer1014.npz
 - CSI2_schaefer400.npz
 - CSI2_schaefer414.npz
 - CSI2_visual_rois.npz

CSI3: 4 npz files
 - CSI3_schaefer1014.npz
 - CSI3_schaefer400.npz
 - CSI3_schaefer414.npz
 - CSI3_visual_rois.npz

CSI4: 4 npz files
 - CSI4_schaefer1014.npz
 - CSI4_schaefer400.npz
 - CSI4_schaefer414.npz
 - CSI4_visual_rois.npz
Inspecting: /home/mariop/Documents/Programming/bold5000-gan/processed_data/CSI1/CSI1_schaefer400.npz
[load_npz] Skipping key 'labels' in CSI1_schaefer400.npz: Object arrays cannot be loaded when allow_pickle=False
Keys (7): ['betas', 'imgnames', 'subject', 'sessions', 'local_idxs', 'dataset_sources', 'parcel_names'] 
                     b

## Verify Data inside .npz files

In [4]:
# --- 0) Notebook setup (paths, reproducibility, display) ---
from __future__ import annotations

import os
from pathlib import Path

import numpy as np

# Root of this repo (assumes this notebook lives at repo root)
REPO_ROOT = Path.cwd()
PROCESSED_DIR = REPO_ROOT / "processed_data"

# Subject IDs present in processed_data/
SUBJECTS = ["CSI1", "CSI2", "CSI3", "CSI4"]

# Common Schaefer parcellations in this repo
SCHAEFER_VARIANTS = [ "schaefer1014"]

# Safer default: prefer NOT unpickling. If an npz contains object arrays,
# we'll either (a) skip those keys or (b) enable pickle if requested.
ALLOW_PICKLE = False
SKIP_OBJECT_ARRAYS = True

print("REPO_ROOT:", REPO_ROOT)
print("PROCESSED_DIR exists:", PROCESSED_DIR.exists())
print("Subjects:", SUBJECTS)
print("Schaefer variants:", SCHAEFER_VARIANTS)

def _npz_path(subject: str, variant: str) -> Path:
    """Return path to a per-subject Schaefer .npz file."""
    return PROCESSED_DIR / subject / f"{subject}_{variant}.npz"


def _visual_rois_path(subject: str) -> Path:
    """Return path to a per-subject visual ROIs .npz file."""
    return PROCESSED_DIR / subject / f"{subject}_visual_rois.npz"


def _require_exists(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing expected file: {path}")

def list_available_npzs(subject: str) -> dict[str, Path]:
    """List available npz files for a given subject."""
    subj_dir = PROCESSED_DIR / subject
    if not subj_dir.exists():
        return {}
    out: dict[str, Path] = {}
    for p in sorted(subj_dir.glob("*.npz")):
        out[p.name] = p
    return out


for s in SUBJECTS:
    files = list_available_npzs(s)
    print(f"\n{s}: {len(files)} npz files")
    for name in list(files)[:10]:
        print(" -", name)
    if len(files) > 10:
        print("   ...")

def load_npz(
    path: Path,
    *,
    allow_pickle: bool | None = None,
    skip_object_arrays: bool | None = None,
) -> dict[str, np.ndarray]:
    """Load npz and return a {key: array} dict (eagerly loaded).

    Some of the repo's .npz files may contain object arrays (e.g., strings).
    By default we avoid pickle; if object arrays are present we skip them.

    Set allow_pickle=True to load everything (less safe, but sometimes needed).
    """
    _require_exists(path)
    if allow_pickle is None:
        allow_pickle = ALLOW_PICKLE
    if skip_object_arrays is None:
        skip_object_arrays = SKIP_OBJECT_ARRAYS

    out: dict[str, np.ndarray] = {}
    with np.load(path, allow_pickle=allow_pickle) as f:
        for k in f.files:
            try:
                arr = f[k]
            except ValueError as e:
                # Typical: "Object arrays cannot be loaded when allow_pickle=False"
                if skip_object_arrays:
                    print(f"[load_npz] Skipping key '{k}' in {path.name}: {e}")
                    continue
                raise

            if skip_object_arrays and isinstance(arr, np.ndarray) and arr.dtype == object:
                print(f"[load_npz] Skipping object array key '{k}' in {path.name}")
                continue
            out[k] = arr
    return out


def describe_npz(npz: dict[str, np.ndarray], max_keys: int = 20) -> None:
    keys = list(npz.keys())
    print(f"Keys ({len(keys)}):", keys[:max_keys], "..." if len(keys) > max_keys else "")
    for k in keys[:max_keys]:
        a = npz[k]
        print(f"  {k:>24}  shape={a.shape}  dtype={a.dtype}")


# Pick one subject+variant to inspect first
SUBJECT = SUBJECTS[0]
VARIANT = SCHAEFER_VARIANTS[0]
PATH = _npz_path(SUBJECT, VARIANT)
print("Inspecting:", PATH)
npz0 = load_npz(PATH)
describe_npz(npz0)

def find_data_array(npz: dict[str, np.ndarray]) -> tuple[str, np.ndarray]:
    """Heuristic: find the 'main' 2D array likely holding betas/voxel/ROI features."""
    # Common candidates
    preferred = [
        "X",
        "data",
        "betas",
        "features",
        "fmri",
        "roi_data",
        "roi_betas",
    ]
    for k in preferred:
        if k in npz and npz[k].ndim >= 2:
            return k, npz[k]

    # Fallback: choose the largest numeric array
    best_k = None
    best = None
    best_size = -1
    for k, a in npz.items():
        if not isinstance(a, np.ndarray):
            continue
        if a.dtype.kind not in "iufb":
            continue
        if a.size > best_size and a.ndim >= 2:
            best_k, best, best_size = k, a, a.size
    if best_k is None:
        raise ValueError("Couldn't find a plausible numeric 2D+ data array in npz")
    return best_k, best


data_key, X = find_data_array(npz0)
print("Main data key:", data_key)
print("X:", X.shape, X.dtype)

# Basic shape contract we want to validate:
# - samples dimension == number of images/trials
# - feature dimension == parcels/ROIs/voxels
print("n_samples:", X.shape[0])
print("n_features:", X.shape[1] if X.ndim >= 2 else None)

REPO_ROOT: /home/mariop/Documents/Programming/bold5000-gan
PROCESSED_DIR exists: True
Subjects: ['CSI1', 'CSI2', 'CSI3', 'CSI4']
Schaefer variants: ['schaefer1014']

CSI1: 4 npz files
 - CSI1_schaefer1014.npz
 - CSI1_schaefer400.npz
 - CSI1_schaefer414.npz
 - CSI1_visual_rois.npz

CSI2: 4 npz files
 - CSI2_schaefer1014.npz
 - CSI2_schaefer400.npz
 - CSI2_schaefer414.npz
 - CSI2_visual_rois.npz

CSI3: 4 npz files
 - CSI3_schaefer1014.npz
 - CSI3_schaefer400.npz
 - CSI3_schaefer414.npz
 - CSI3_visual_rois.npz

CSI4: 4 npz files
 - CSI4_schaefer1014.npz
 - CSI4_schaefer400.npz
 - CSI4_schaefer414.npz
 - CSI4_visual_rois.npz
Inspecting: /home/mariop/Documents/Programming/bold5000-gan/processed_data/CSI1/CSI1_schaefer1014.npz
[load_npz] Skipping key 'labels' in CSI1_schaefer1014.npz: Object arrays cannot be loaded when allow_pickle=False
Keys (7): ['betas', 'imgnames', 'subject', 'sessions', 'local_idxs', 'dataset_sources', 'parcel_names'] 
                     betas  shape=(5254, 1014)  dt

## Map Trial to Image

In [9]:
# --- 1) Match Schaefer1014 trials to raw stimulus images ---
# Goal: given an imgnames entry (e.g., "n01930112_19568.JPEG"), return the on-disk path.

import glob
from functools import lru_cache

# This should match the constant used in `CLIP_Data_Pipeline.ipynb`
STIMULI_DIR = "/media/hdd/BOLD5000/BOLD5000_Stimuli/Scene_Stimuli/Presented_Stimuli"


def reconstruct_raw_imagename(name) -> str:
    """Normalize imgnames entries to a base filename.

    In BOLD5000-style metadata this is usually already something like:
      - 'n01930112_19568.JPEG'
      - 'COCO_train2014_000000123456.jpg'

    This function strips folders if they appear and ensures it's a plain basename.
    """
    # Handle bytes (common in npz allow_pickle=True)
    if isinstance(name, (bytes, np.bytes_)):
        name = name.decode("utf-8", errors="replace")
    name = str(name)
    return os.path.basename(name)


@lru_cache(maxsize=1)
def _build_image_map(stimuli_dir: str = STIMULI_DIR) -> dict[str, str]:
    """Index all image files under the stimuli directory.

    Returns:
        dict mapping basename -> absolute path
    """
    if not os.path.exists(stimuli_dir):
        raise FileNotFoundError(
            "STIMULI_DIR not found. Update STIMULI_DIR to where BOLD5000 stimuli live on this machine. "
            f"Got: {stimuli_dir}"
        )

    image_paths = (
        glob.glob(f"{stimuli_dir}/**/*.jpg", recursive=True)
        + glob.glob(f"{stimuli_dir}/**/*.jpeg", recursive=True)
        + glob.glob(f"{stimuli_dir}/**/*.JPEG", recursive=True)
        + glob.glob(f"{stimuli_dir}/**/*.png", recursive=True)
    )

    # basename -> full path (if collisions exist, last one wins; we'll warn below)
    image_map: dict[str, str] = {}
    collisions = 0
    for p in image_paths:
        b = os.path.basename(p)
        if b in image_map and image_map[b] != p:
            collisions += 1
        image_map[b] = p

    if collisions:
        print(f"[warn] basename collisions encountered while indexing: {collisions}")

    print(f"Indexed {len(image_paths)} images under {stimuli_dir}")
    return image_map


def resolve_raw_image_path(imagename: str, *, stimuli_dir: str = STIMULI_DIR) -> str:
    """Deliverable: given an image name, return the absolute raw image path.

    Args:
        imagename: string or bytes name from the fMRI metadata; may include folders.
        stimuli_dir: root directory containing Presented_Stimuli (defaults to repo convention).

    Returns:
        Absolute path on disk.

    Raises:
        FileNotFoundError if not found.
    """
    base = reconstruct_raw_imagename(imagename)
    image_map = _build_image_map(stimuli_dir)
    if base not in image_map:
        raise FileNotFoundError(
            f"Image '{base}' not found under stimuli_dir='{stimuli_dir}'. "
            "If the drive is mounted elsewhere, update STIMULI_DIR."
        )
    return image_map[base]


# --- Sanity check: do Schaefer1014 NPZs contain imgnames? ---
# We load with allow_pickle=True just for metadata keys like imgnames.

SUBJECT_FOR_IMG_CHECK = SUBJECTS[0]
S1014 = load_npz(_npz_path(SUBJECT_FOR_IMG_CHECK, "schaefer1014"), allow_pickle=True, skip_object_arrays=False)

img_key_candidates = ["imgnames", "image_names", "images", "stimuli", "stim_names"]
img_key = next((k for k in img_key_candidates if k in S1014), None)

if img_key is None:
    print("Couldn't find an image-name key in the schaefer1014 npz.")
    print("Available keys:", list(S1014.keys())[:50])
else:
    imgnames = S1014[img_key]
    print(f"Found image names under key '{img_key}' with shape {imgnames.shape} dtype={imgnames.dtype}")
    sample = reconstruct_raw_imagename(imgnames[0])
    print("Sample reconstructed name:", sample)

    # Try resolving a single image path (will raise if STIMULI_DIR isn't correct/mounted)
    try:
        p = resolve_raw_image_path(sample)
        print("Resolved path:", p)
    except FileNotFoundError as e:
        print("Resolve failed (expected if stimuli drive isn't mounted here):")
        print(" ", e)


Found image names under key 'imgnames' with shape (5254,) dtype=<U31
Sample reconstructed name: n01930112_19568.JPEG
Indexed 4916 images under /media/hdd/BOLD5000/BOLD5000_Stimuli/Scene_Stimuli/Presented_Stimuli
Resolved path: /media/hdd/BOLD5000/BOLD5000_Stimuli/Scene_Stimuli/Presented_Stimuli/ImageNet/n01930112_19568.JPEG


# CLIP

In [11]:
# --- 2) CLIP: image filepath -> embedding ---
# Deliverable: clip_embed_image(image_path) -> np.ndarray

from functools import lru_cache

import numpy as np
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPVisionModelWithProjection


@lru_cache(maxsize=4)
def _load_clip(model_id: str = "openai/clip-vit-large-patch14", device_str: str | None = None):
    """Load CLIP model + processor once per kernel.

    device_str:
      - None: auto (cuda if available else cpu)
      - 'cpu' or 'cuda': force
    """
    if device_str is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    else:
        device = torch.device(device_str)

    processor = CLIPProcessor.from_pretrained(model_id)
    model = CLIPVisionModelWithProjection.from_pretrained(model_id).to(device)
    model.eval()
    return processor, model, device


def clip_embed_image(
    image_path: str,
    *,
    model_id: str = "openai/clip-vit-large-patch14",
    device: str | None = None,
    fallback_to_cpu: bool = True,
) -> np.ndarray:
    """Compute a CLIP image embedding from a local image filepath.

    Args:
        image_path: absolute path to an image file.
        model_id: HF model id; default matches `CLIP_Data_Pipeline.ipynb`.
        device: None|'cuda'|'cpu'. None auto-selects.
        fallback_to_cpu: if True, retry on CPU when a CUDA error occurs.

    Returns:
        embedding: shape (768,) float32 numpy array for ViT-L/14.
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image path does not exist: {image_path}")

    def _run(dev: str | None) -> np.ndarray:
        processor, model, torch_device = _load_clip(model_id, dev)
        img = Image.open(image_path).convert("RGB")
        inputs = processor(images=img, return_tensors="pt")
        inputs = {k: v.to(torch_device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            emb = outputs.image_embeds.squeeze(0).detach().cpu().numpy().astype(np.float32)
        return emb

    try:
        return _run(device)
    except RuntimeError as e:
        msg = str(e)
        if fallback_to_cpu and ("CUDA" in msg or "cuda" in msg) and (device is None or device == "cuda"):
            # Best-effort cleanup then retry on CPU
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
            return _run("cpu")
        raise


# --- Optional smoke test: first Schaefer1014 image -> path -> CLIP embedding ---
if 'imgnames' in globals() and len(imgnames) > 0:
    try:
        first_name = reconstruct_raw_imagename(imgnames[0])
        first_path = resolve_raw_image_path(first_name)
        emb = clip_embed_image(first_path)
        print("first_name:", first_name)
        print("first_path:", first_path)
        print("embedding shape:", emb.shape, "dtype:", emb.dtype, "norm:", float(np.linalg.norm(emb)))
    except FileNotFoundError as e:
        print("Smoke test skipped (image not found / stimuli not mounted):")
        print(" ", e)


Loading weights: 100%|██████████| 392/392 [00:00<00:00, 9619.18it/s]
CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.embeddings.position_ids                         | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.b

first_name: n01930112_19568.JPEG
first_path: /media/hdd/BOLD5000/BOLD5000_Stimuli/Scene_Stimuli/Presented_Stimuli/ImageNet/n01930112_19568.JPEG
embedding shape: (768,) dtype: float32 norm: 17.427244186401367


## Create ClipEmbeddings.json

In [12]:
# --- 3) Build ClipEmbeddings.json (imgname -> path + CLIP embedding + labels) ---

import json
from collections import defaultdict

OUT_JSON = REPO_ROOT / "ClipEmbeddings.json"
OUT_JSON_TMP = REPO_ROOT / "ClipEmbeddings.partial.json"
SEMANTIC_LABELS_JSON = REPO_ROOT / "SemanticLabels.json"

# Set this True if you want to overwrite existing output.
FORCE_REBUILD = False

# For reliability, default to CPU for the full build (GPU can be faster but may error on long runs).
# You can set to 'cuda' if you want, and it will still fallback-to-CPU on CUDA errors.
EMBED_DEVICE: str | None = "cpu"  # 'cpu' | 'cuda' | None

# Save partial progress every N records
CHECKPOINT_EVERY = 100

if OUT_JSON.exists() and not FORCE_REBUILD:
    print(f"{OUT_JSON} already exists. Set FORCE_REBUILD=True to regenerate.")
else:
    # 1) Load semantic label mapping (coarse category -> {aliases, labels})
    with open(SEMANTIC_LABELS_JSON, "r") as f:
        semantic_map = json.load(f)

    # Build: fine_label -> [coarse_categories]
    fine_to_coarse: dict[str, list[str]] = defaultdict(list)
    for coarse, payload in semantic_map.items():
        for fine in payload.get("labels", []):
            fine_to_coarse[str(fine)].append(coarse)


    def human_labels_for_imgname(imgname: str) -> list[str]:
        """Best-effort 'human readable label(s)' for an image.

        We don't have a single authoritative per-image label file in this repo snapshot.
        So we provide a multi-label list using:
          - dataset hint from filename (COCO/ImageNet/Scene)
          - semantic coarse categories if the imgname contains a known fine label token

        If later you add a real per-image label table, we can swap this out.
        """
        base = reconstruct_raw_imagename(imgname)
        out: list[str] = []

        # Dataset hint
        if base.startswith("COCO_"):
            out.append("coco")
        elif base.lower().endswith(".jpeg") and base.startswith("n"):
            out.append("imagenet")
        else:
            out.append("scene")

        # Token-based coarse category guess (very conservative)
        name_no_ext = os.path.splitext(base)[0]
        tokens = set(name_no_ext.replace("-", "_").split("_"))
        for t in list(tokens):
            if t in fine_to_coarse:
                out.extend(fine_to_coarse[t])

        # De-dupe while preserving order
        seen = set()
        out2 = []
        for x in out:
            if x not in seen:
                seen.add(x)
                out2.append(x)
        return out2


    # 2) Get all image names from Schaefer1014 metadata across subjects.
    all_imgnames = []
    for s in SUBJECTS:
        meta = load_npz(_npz_path(s, "schaefer1014"), allow_pickle=True, skip_object_arrays=False)
        if "imgnames" not in meta:
            raise KeyError(f"Expected 'imgnames' in {s} schaefer1014 npz, found keys={list(meta.keys())}")
        all_imgnames.append(meta["imgnames"])

    all_imgnames = np.concatenate(all_imgnames)
    unique_imgnames = sorted(set(reconstruct_raw_imagename(x) for x in all_imgnames))
    print("Total presentations across subjects:", int(all_imgnames.shape[0]))
    print("Unique image names:", len(unique_imgnames))


    # 3) Build JSON entries
    def _to_serializable_floats(x: np.ndarray) -> list[float]:
        x = np.asarray(x, dtype=np.float32).reshape(-1)
        return x.tolist()


    records = []
    missing = []

    for i, name in enumerate(unique_imgnames):
        try:
            path = resolve_raw_image_path(name)
        except FileNotFoundError:
            missing.append(name)
            continue

        emb = clip_embed_image(path, device=EMBED_DEVICE, fallback_to_cpu=True)
        records.append(
            {
                "imgname": name,
                "path": path,
                "clip_embedding": _to_serializable_floats(emb),
                # multi-label list by design
                "labels": human_labels_for_imgname(name),
            }
        )

        if CHECKPOINT_EVERY and (len(records) % CHECKPOINT_EVERY == 0):
            payload_partial = {
                "model_id": "openai/clip-vit-large-patch14",
                "embedding_dim": 768,
                "stimuli_dir": STIMULI_DIR,
                "n_unique_images": len(unique_imgnames),
                "n_records": len(records),
                "missing": missing,
                "items": records,
                "partial": True,
            }
            with open(OUT_JSON_TMP, "w") as f:
                json.dump(payload_partial, f)
            print(f"Checkpoint: {len(records)} records -> {OUT_JSON_TMP}")

    print("Records written:", len(records))
    print("Missing images:", len(missing))
    if missing:
        print("First 10 missing:", missing[:10])


    # 4) Write final
    payload = {
        "model_id": "openai/clip-vit-large-patch14",
        "embedding_dim": 768,
        "stimuli_dir": STIMULI_DIR,
        "n_unique_images": len(unique_imgnames),
        "n_records": len(records),
        "missing": missing,
        "items": records,
        "partial": False,
    }

    with open(OUT_JSON, "w") as f:
        json.dump(payload, f)

    # remove partial if it exists
    try:
        if OUT_JSON_TMP.exists():
            OUT_JSON_TMP.unlink()
    except Exception:
        pass

    print("Saved:", OUT_JSON)


Total presentations across subjects: 18870
Unique image names: 4916
Checkpoint: 100 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Checkpoint: 100 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Checkpoint: 200 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Checkpoint: 200 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Checkpoint: 300 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Checkpoint: 300 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Checkpoint: 400 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Checkpoint: 400 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Checkpoint: 500 records -> /home/mariop/Documents/Programming/bold5000-gan/ClipEmbeddings.partial.json
Check